# Exercise 6 - Simulating Survey Responses
_By Georg Ahnert_

In this exercise we will look at how LLMs can be used to generate synthetic survey responses. We will also look into how to evaluate them against human survey data.

**You will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of three parts:
1. Persona simulation
2. Generating closed-ended responses
2. Evaluation against human survey data

### Setup

Follow the instructions from Exercise 3 for LLM setup. Make sure you are using the correct kernel is selected for this notebook.

In [ ]:
%pip install vllm
%pip install scikit-learn

## 1. Persona simulation

Surveys are often simulated _in-silico_ with LLMs on an individual level. We provide the LLM with a description of a persona to predict the survey responses of the same individual.

Here, we will use data from the American National Election Study (ANES) to simulate a U.S. population. This closely follows the work of [Argyle et al. (2023)](https://www.cambridge.org/core/journals/political-analysis/article/out-of-one-many-using-language-models-to-simulate-human-samples/035D7C8A55B237942FB6DBAD7CAA4E49).

First, let's load a subset of the ANES 2016 dataset. For more detail on the ANES, see https://electionstudies.org/data-center/2016-time-series-study/

In [1]:
import pandas as pd

argyle_df = pd.read_csv('2016_anes_argyle.csv')
argyle_df

,race,discuss_politics,ideology,party,church_goer,age,gender,political_interest,patriotism,state,ground_truth
0,white,Yes,NaN,a strong Republican,attend church,29.0,man,somewhat,extremely good,Louisiana,Trump
1,white,Yes,slightly conservative,a weak Republican,do not attend church,26.0,man,very,extremely good,Arkansas,Trump
2,white,No,NaN,an independent who leans Democratic,do not attend church,23.0,man,not very,moderately good,Mississippi,Non-voter
3,white,Yes,NaN,an independent who leans Republican,attend church,58.0,man,somewhat,extremely good,Tennessee,Trump
4,white,No,moderate,an independent who leans Democratic,attend church,38.0,woman,not very,extremely good,Ohio,Non-voter
...,...,...,...,...,...,...,...,...,...,...,...
4265,white,Yes,NaN,a strong democrat,attend church,37.0,woman,not very,extremely good,Virginia,Clinton
4266,white,Yes,moderate,a strong democrat,attend church,82.0,man,somewhat,extremely good,Virginia,Trump
4267,white,No,moderate,a weak Democrat,attend church,27.0,man,not very,a little good,Georgia,Non-voter
4268,black,Yes,conservative,a strong democrat,attend church,39.0,woman,somewhat,neither good nor bad,North Carolina,Clinton


As you can see, we have both sociodemographic variables (e.g. age, gender) and attitudes (e.g., ideology), as well as a "ground_truth" column of self-reported vote choice.

Note that some individuals have _item-level missingness_, i.e., did not respond to all questions. The survey also has _person-level missingness_, i.e., some people that have been originally sampled did not respond to the survey at all.

Still, the ANES tries to sample survey participants that can represent the population of eligible voters in the U.S.

We will work with a small, random subsample of this for this exercise.

In [55]:
sample_df = argyle_df.sample(n=50, random_state=42)
sample_df.head()

,race,discuss_politics,ideology,party,church_goer,age,gender,political_interest,patriotism,state,ground_truth
1703,hispanic,Yes,conservative,a strong Republican,attend church,42.0,man,somewhat,moderately good,Oklahoma,Trump
1173,asian,NaN,slightly conservative,a weak Republican,attend church,36.0,woman,NaN,NaN,Illinois,Non-voter
308,NaN,Yes,moderate,an independent who leans Democratic,NaN,76.0,man,somewhat,extremely good,Tennessee,Clinton
1322,white,Yes,NaN,an independent who leans Republican,do not attend church,67.0,woman,not very,a little good,Texas,Non-voter
3271,NaN,NaN,NaN,a weak Democrat,attend church,52.0,man,NaN,NaN,Georgia,Non-voter


There are a couple of commonly used approaches to simulating personas (see also https://arxiv.org/abs/2507.16076).

Let's try a key-value format first:

In [12]:
personas = []

for id, individual in sample_df.iterrows():
    valid_responses = individual.dropna() # remove item-level missingness from the simulation
    persona_attributes = valid_responses.drop('ground_truth') # remove the column that we want to predict
    personas.append(f"Pretend you are the following person: {persona_attributes.to_dict()}")

print(personas[0])
print(personas[4])

Pretend you are the following person: {'race': 'hispanic', 'discuss_politics': 'Yes', 'ideology': 'conservative', 'party': 'a strong Republican', 'church_goer': 'attend church', 'age': 42.0, 'gender': 'man', 'political_interest': 'somewhat', 'patriotism': 'moderately good', 'state': 'Oklahoma'}
Pretend you are the following person: {'party': 'a weak Democrat', 'church_goer': 'attend church', 'age': 52.0, 'gender': 'man', 'state': 'Georgia'}


Another way to implement personas would be "interview-style" question-answer pairs:

In [11]:
personas = []

for id, individual in sample_df.iterrows():
    persona = ''
    if pd.notna(individual.race): # only add existing data
        persona += f"Interviewer: What is your race?\nInterviewee: I am {individual.race.title()}.\n"
    if pd.notna(individual.age):
        persona += f"Interviewer: How old are you?\nInterviewee: I am {int(individual.age)} years old.\n"
    if pd.notna(individual.political_interest):
        persona += f"Interviewer: Are you interested in politics?\nInterviewee: I am {individual.political_interest} interested in politics.\n"
    personas.append(persona)

print(personas[0])
print(personas[4])

Interviewer: What is your race?
Interviewee: I am Hispanic.
Interviewer: How old are you?
Interviewee: I am 42 years old.
Interviewer: Are you interested in politics?
Interviewee: I am somewhat interested in politics.

Interviewer: How old are you?
Interviewee: I am 52 years old.



### Task 1

[Argyle et al. (2023)](https://www.cambridge.org/core/journals/political-analysis/article/out-of-one-many-using-language-models-to-simulate-human-samples/035D7C8A55B237942FB6DBAD7CAA4E49) used "biography" prompt format. Have a look at the supplementary material of their paper and re-implement the prompt format of study 2 using the data stored in `sample_df`.

In [13]:
personas = []

for id, row in sample_df.iterrows():
    persona = ""
    if pd.notna(row.race):
        persona += f"Racially, I am {row.race}. "
    if pd.notna(row.discuss_politics):
        if row.discuss_politics == 'Yes':
            persona += "I like to discuss politics with my family and friends. "
        else:
            persona += "I never discuss politics with my family or friends. "
    if pd.notna(row.ideology):
        persona += f"Ideologically, I am {row.ideology}. "
    if pd.notna(row.party):
        persona += f"Politically, I am an {row.party}. "
    if pd.notna(row.church_goer):
        persona += f"I {row.church_goer}. "
    if pd.notna(row.age):
        persona += f"I am {int(row.age)} years old. "
    if pd.notna(row.gender):
        persona += f"I am a {row.gender}. "
    if pd.notna(row.political_interest):
        persona += f"I am {row.political_interest} interested in politics. "
    if pd.notna(row.patriotism):
        persona += f"It makes me feel {row.patriotism} to see the American flag. "
    if pd.notna(row.state):
        persona += f"I am from {row.state}. "
    personas.append(persona)

print(personas[0])
print(personas[4])

Racially, I am hispanic. I like to discuss politics with my family and friends. Ideologically, I am conservative. Politically, I am an a strong Republican. I attend church. I am 42 years old. I am a man. I am somewhat interested in politics. It makes me feel moderately good to see the American flag. I am from Oklahoma. 
Politically, I am an a weak Democrat. I attend church. I am 52 years old. I am a man. I am from Georgia. 


## 2. Generating closed-ended responses

Now that we have personas implemented, let's actually try to predict survey answers. First, let's set up the model. We'll use Qwen 3 4B here, since it's a bit better at instruction following than OLMo 2 1B.

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")

Let's use the same prompt as Argyle et al. (2023) to try to predict vote choice:

In [25]:
from transformers import set_seed

set_seed(42)

batch_inputs = [
    [
        {
            "role": "user",
            "content": f"{persona}\nIn the 2016 presidential election, I voted for"
        }
    ]
    for persona in personas[:2]
]
    
outputs = pipe(batch_inputs, max_new_tokens=100)

In [26]:
for conversation in outputs:
    reply = conversation[0]['generated_text'][-1]['content']
    print('\n---\n')
    print(reply)


---

In the 2016 presidential election, you voted for Donald J. Trump.  

This conclusion is based on the information you provided:  
- You are a Hispanic man from Oklahoma.  
- You are a strong Republican.  
- You are ideologically conservative and politically aligned with the Republican Party.  
- You are somewhat interested in politics and feel positively about the American flag, which is often associated with Republican values.  

While Hispanic voters in Oklahoma have historically shown a mix of support across

---

Based on the information you've provided—being Asian, slightly conservative ideologically, a weak Republican, attending church, 36 years old, a woman from Illinois—it's likely that in the 2016 U.S. presidential election, you voted for **Hillary Clinton**.

Here’s why:

- While you identify as a "weak Republican" and slightly conservative, your political leaning is not strongly aligned with the Republican Party. Many women, especially those from moderate or suburban ba

We observe that the model is trained to produce open-ended text instead of responding with a closed-ended survey response. But maybe some instruction in the system prompt can help.

In [39]:
batch_inputs = [
    [
        {
            "role": "system",
            "content": "You are a political scientist that predicts survey responses. " + \
                       "The possible answer options are: ['Clinton', 'Trump', 'Non-voter']. " + \
                        "Only respond with the most likely answer option. Do not produce any additional text!"
        },
        {
            "role": "user",
            "content": f"{persona}\nIn the 2016 presidential election, I voted for"
        }
    ]
    for persona in personas
]
    
single_outputs = pipe(batch_inputs, max_new_tokens=100)

In [40]:
for conversation in single_outputs[:10]:
    reply = conversation[0]['generated_text'][-1]['content']
    print(reply)

Trump
Clinton
Clinton
Trump
Trump
Clinton
Trump
Clinton
Clinton
Trump


### Task 2

An alternative to a single label is to let the model generate a probability distribution across the answer options for each individual. See for instance [Meister et al. (2025)](https://aclanthology.org/2025.naacl-long.2/) or [Ahnert et al. (2025)](https://arxiv.org/abs/2510.11586). Implement this "Verbalized Distribution" method using the following JSON format:

```
{
    "Clinton": <probability>,
    "Trump": <probability>,
    "Non-voter": <probability>
}
```

In [41]:
output_format = """
{
    "Clinton": <probability>,
    "Trump": <probability>,
    "Non-voter": <probability>
}
"""

batch_inputs = [
    [
        {
            "role": "system",
            "content": "You are a political scientist that predicts survey responses. " + \
                       "The possible answer options are: ['Clinton', 'Trump', 'Non-voter']. " + \
                        "Only respond with a probability for each answer option in the following JSON format: " + \
                        output_format + \
                        "Do not produce any additional text!"
        },
        {
            "role": "user",
            "content": f"{persona}\nIn the 2016 presidential election, I voted for"
        }
    ]
    for persona in personas
]
    
distr_outputs = pipe(batch_inputs, max_new_tokens=100)

In [42]:
for conversation in distr_outputs[:2]:
    reply = conversation[0]['generated_text'][-1]['content']
    print(reply)

{
    "Clinton": 0.05,
    "Trump": 0.92,
    "Non-voter": 0.03
}
{
    "Clinton": 0.75,
    "Trump": 0.20,
    "Non-voter": 0.05
}


## 3. Evaluation against human survey data

Now, let's see how well we were actually able to predict survey responses. Let's start with an individual-level evaluation first:

In [47]:
from sklearn.metrics import classification_report

predicted_labels = [conversation[0]['generated_text'][-1]['content'] for conversation in single_outputs]
predicted_labels[:10]

['Trump',
 'Clinton',
 'Clinton',
 'Trump',
 'Trump',
 'Clinton',
 'Trump',
 'Clinton',
 'Clinton',
 'Trump']

In [48]:
true_labels = sample_df['ground_truth'].to_list()
true_labels[:10]

['Trump',
 'Non-voter',
 'Clinton',
 'Non-voter',
 'Non-voter',
 'Trump',
 'Non-voter',
 'Clinton',
 'Trump',
 'Non-voter']

In [53]:
print(classification_report(
    true_labels,
    predicted_labels,
    labels=['Trump', 'Clinton', 'Non-voter'] # ignore additional labels that the LLM hallucinated
))

              precision    recall  f1-score   support

       Trump       0.48      0.72      0.58        18
     Clinton       0.41      0.90      0.56        10
   Non-voter       1.00      0.05      0.09        22

    accuracy                           0.46        50
   macro avg       0.63      0.56      0.41        50
weighted avg       0.70      0.46      0.36        50



We are reasonably good with Trump and Clinton voters, but predicting Non-voters is hard.

Another way to look at evaluation is a distribution match. Do we get the response distributions in subpopulations correct? Let's have a look at the people who do and do not go to church.

In [67]:
result_df = sample_df.copy()
result_df['prediction'] = predicted_labels

gt_distr = result_df.groupby('church_goer')['ground_truth'].value_counts(normalize=True)
gt_distr

church_goer           ground_truth
attend church         Trump           0.419355
                      Non-voter       0.354839
                      Clinton         0.225806
do not attend church  Non-voter       0.611111
                      Trump           0.277778
                      Clinton         0.111111
Name: proportion, dtype: float64

In [80]:
pred_distr = result_df.groupby('church_goer')['prediction'].value_counts(normalize=True)
pred_distr # NOTE that Non-voters have never been predicted for the "attend church" group!

church_goer           prediction
attend church         Trump         0.580645
                      Clinton       0.419355
do not attend church  Trump         0.500000
                      Clinton       0.444444
                      Non-voter     0.055556
Name: proportion, dtype: float64

To measure similarity between these two distributions, we can calculate [total variation distance](https://en.wikipedia.org/wiki/Total_variation_distance_of_probability_measures) and [Jensen-Shannon Divergence](https://en.wikipedia.org/wiki/Jensen%E2%80%93Shannon_divergence) as follows:

In [128]:
import numpy as np
from scipy.spatial.distance import jensenshannon

# Make sure that all labels (i.e., also Non-voters) are found in all groups in the same order
index = ['Trump', 'Non-voter', 'Clinton']

pred_labels = pred_distr.reset_index(level=0)
gt_labels = gt_distr.reset_index(level=0)
display(gt_labels)

for subpopulation in ['attend church', 'do not attend church']:
    subpop_pred = pred_labels[pred_labels.church_goer == subpopulation].reindex(index=index).fillna(0)
    subpop_gt = gt_labels[gt_labels.church_goer == subpopulation].reindex(index=index).fillna(0)
    
    tv_distance = 0.5 * np.abs(subpop_gt.proportion.to_numpy() - subpop_pred.proportion.to_numpy()).sum()
    print('---')
    print(f"Total Variation Distance for '{subpopulation}': {tv_distance:.3f} (lower is better)")

    js_divergence = jensenshannon(subpop_gt.proportion.to_numpy(), subpop_pred.proportion.to_numpy()) ** 2 # returns sqrt(JSD) by default
    print(f"Jensen-Shannon Divergence for '{subpopulation}': {js_divergence:.3f} (lower is better)")

,church_goer,proportion
ground_truth,,
Trump,attend church,0.419355
Non-voter,attend church,0.354839
Clinton,attend church,0.225806
Non-voter,do not attend church,0.611111
Trump,do not attend church,0.277778
Clinton,do not attend church,0.111111


---
Total Variation Distance for 'attend church': 0.355 (lower is better)
Jensen-Shannon Divergence for 'attend church': 0.144 (lower is better)
---
Total Variation Distance for 'do not attend church': 0.556 (lower is better)
Jensen-Shannon Divergence for 'do not attend church': 0.205 (lower is better)


### Task 3

How does the Verbalized Distribution output from Task 2 evaluate on subpopulation-level distributions?

1. Parse the JSON responses from Task 2
2. Calculate the mean response distribution in each subpopulation
3. Calculate Total Variation Distance and Jensen-Shannon Divergence for both subpopulations

In [117]:
### Parse JSON responses from Task 2
import json
import warnings

results = []

for conversation in distr_outputs:
    response = conversation[0]['generated_text'][-1]['content']
    try:
        results.append(json.loads(response))
    except:
        warnings.warn(f"Cannot parse response: {response}")
        results.append({})

distr_df = pd.DataFrame(results)
distr_df.index = sample_df.index
distr_df.head()

,Clinton,Trump,Non-voter
1703,0.05,0.92,0.03
1173,0.75,0.20,0.05
308,0.75,0.20,0.05
1322,0.15,0.80,0.05
3271,0.75,0.20,0.05


In [124]:
# Combine with original survey data
distr_result_df = pd.merge(sample_df, distr_df, left_index=True, right_index=True)

# Make sure that the probabilities sum up to 1
response_options = ['Clinton', 'Trump', 'Non-voter']
distr_result_df['prob_sum'] = distr_result_df[response_options].sum(axis=1)
for option in response_options:
    distr_result_df[option] = distr_result_df[option] / distr_result_df['prob_sum']

distr_result_df.head()

,race,discuss_politics,ideology,party,church_goer,age,gender,political_interest,patriotism,state,ground_truth,Clinton,Trump,Non-voter,prob_sum
1703,hispanic,Yes,conservative,a strong Republican,attend church,42.0,man,somewhat,moderately good,Oklahoma,Trump,0.05,0.92,0.03,1.0
1173,asian,NaN,slightly conservative,a weak Republican,attend church,36.0,woman,NaN,NaN,Illinois,Non-voter,0.75,0.20,0.05,1.0
308,NaN,Yes,moderate,an independent who leans Democratic,NaN,76.0,man,somewhat,extremely good,Tennessee,Clinton,0.75,0.20,0.05,1.0
1322,white,Yes,NaN,an independent who leans Republican,do not attend church,67.0,woman,not very,a little good,Texas,Non-voter,0.15,0.80,0.05,1.0
3271,NaN,NaN,NaN,a weak Democrat,attend church,52.0,man,NaN,NaN,Georgia,Non-voter,0.75,0.20,0.05,1.0


In [127]:
### Mean response distributions in each subpopulation

mean_distr_df = distr_result_df.groupby('church_goer')[response_options].mean().reset_index()
mean_distr_df

,church_goer,Clinton,Trump,Non-voter
0,attend church,0.482903,0.487419,0.029677
1,do not attend church,0.573333,0.395000,0.031667


In [134]:
### Total Variation Distance and Jensen-Shannon Divergence
gt_labels = gt_distr.reset_index(level=0)

for subpopulation in ['attend church', 'do not attend church']:
    subpop_gt = gt_labels[gt_labels.church_goer == subpopulation].reindex(index=index).fillna(0)
    subpop_pred = mean_distr_df[mean_distr_df.church_goer == subpopulation][response_options]
    
    tv_distance = 0.5 * np.abs(subpop_gt.proportion.to_numpy() - subpop_pred.iloc[0].to_numpy()).sum()
    print('---')
    print(f"Total Variation Distance for '{subpopulation}': {tv_distance:.3f} (lower is better)")

    js_divergence = jensenshannon(subpop_gt.proportion.to_numpy(), subpop_pred.iloc[0].to_numpy()) ** 2 # returns sqrt(JSD) by default
    print(f"Jensen-Shannon Divergence for '{subpopulation}': {js_divergence:.3f} (lower is better)")

---
Total Variation Distance for 'attend church': 0.196 (lower is better)
Jensen-Shannon Divergence for 'attend church': 0.049 (lower is better)
---
Total Variation Distance for 'do not attend church': 0.296 (lower is better)
Jensen-Shannon Divergence for 'do not attend church': 0.050 (lower is better)


### Task 4 (Optional)

Re-run the experiments with different persona prompt formats and answer scales. You can, for instance, try a "multiple choice question" scale that uses ["A", "B", "C"] as the possible answer options, or you can reverse the order in which the answer options are presented. What do you observe in terms of individual-level macro avg. F1-score and in terms of subpopulation level Total Variation Distance? How far to the outcomes differ between the simulation specifications? Which specification yields the best results?